# Salary Prediction with Machine Learning

This notebook implements multiple machine learning regression algorithms to predict salaries based on features such as Age, Gender, Education Level, Job Title, and Years of Experience.

## Models Implemented
- XGBRegressor
- Extra Trees Regressor
- Decision Tree Regressor
- K-Nearest Neighbors Regressor
- Gradient Boosting Regressor
- Lasso Regression
- Linear Regression
- Ridge Regression
- ElasticNet Regression

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

print('Libraries imported successfully!')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('Salary_Data.csv')
print('Dataset Shape:', df.shape)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print('Dataset Info:')
df.info()
print('\nMissing Values:')
print(df.isnull().sum())

In [ ]:
print('Statistical Summary:')
df.describe()

In [ ]:
# Distribution of categorical features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df['Gender'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('Gender Distribution')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Count')

df['Education Level'].value_counts().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Education Level Distribution')
axes[1].set_xlabel('Education Level')
axes[1].set_ylabel('Count')

df['Salary'].hist(bins=50, ax=axes[2], color='steelblue', edgecolor='black')
axes[2].set_title('Salary Distribution')
axes[2].set_xlabel('Salary')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Average salary by Education Level
plt.figure(figsize=(10, 5))
edu_salary = df.groupby('Education Level')['Salary'].mean().sort_values()
edu_salary.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Average Salary by Education Level')
plt.xlabel('Education Level')
plt.ylabel('Average Salary ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric features)
plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Drop rows with missing values
df = df.dropna()
print('Shape after dropping NaN:', df.shape)

# Encode categorical variables
le = LabelEncoder()

df['Gender'] = le.fit_transform(df['Gender'])

# Map Education Level to ordinal values
education_map = {
    'High School': 0,
    "Bachelor's": 1,
    "Master's": 2,
    'PhD': 3
}
df['Education Level'] = df['Education Level'].map(education_map)

# Encode Job Title
df['Job Title'] = le.fit_transform(df['Job Title'])

print('Preprocessing complete!')
df.head()

## 5. Feature Selection & Train-Test Split

In [ ]:
X = df.drop('Salary', axis=1)
y = df['Salary']

print('Features shape:', X.shape)
print('Target shape:', y.shape)
print('\nFeatures:')
print(X.columns.tolist())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training set size:', X_train.shape)
print('Test set size:', X_test.shape)

In [ ]:
# Feature Scaling for linear models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Model Training & Evaluation

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    r2 = r2_score(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae = mean_absolute_error(y_te, y_pred)
    return {'Model': name, 'R² Score': round(r2, 6), 'RMSE': round(rmse, 2), 'MAE': round(mae, 2)}

results = []

### XGBoost Regressor

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    verbosity=0
)
result = evaluate_model('XGBRegressor', xgb_model, X_train, X_test, y_train, y_test)
results.append(result)
print(f"XGBRegressor - R²: {result['R² Score']}, RMSE: {result['RMSE']}, MAE: {result['MAE']}")

### Extra Trees Regressor

In [ ]:
et_model = ExtraTreesRegressor(n_estimators=100, random_state=42)
result = evaluate_model('Extra Tree', et_model, X_train, X_test, y_train, y_test)
results.append(result)
print(f"Extra Tree - R²: {result['R² Score']}, RMSE: {result['RMSE']}, MAE: {result['MAE']}")

### Decision Tree Regressor

In [ ]:
dt_model = DecisionTreeRegressor(random_state=42)
result = evaluate_model('Decision Tree', dt_model, X_train, X_test, y_train, y_test)
results.append(result)
print(f"Decision Tree - R²: {result['R² Score']}, RMSE: {result['RMSE']}, MAE: {result['MAE']}")

### K-Nearest Neighbors Regressor

In [ ]:
knn_model = KNeighborsRegressor(n_neighbors=5)
result = evaluate_model('KNeighborsRegressor', knn_model, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(result)
print(f"KNeighborsRegressor - R²: {result['R² Score']}, RMSE: {result['RMSE']}, MAE: {result['MAE']}")

### Gradient Boosting Regressor

In [ ]:
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
result = evaluate_model('Gradient Boosting', gb_model, X_train, X_test, y_train, y_test)
results.append(result)
print(f"Gradient Boosting - R²: {result['R² Score']}, RMSE: {result['RMSE']}, MAE: {result['MAE']}")

### Lasso Regression

In [ ]:
lasso_model = Lasso(alpha=1.0, random_state=42)
result = evaluate_model('Lasso', lasso_model, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(result)
print(f"Lasso - R²: {result['R² Score']}, RMSE: {result['RMSE']}, MAE: {result['MAE']}")

### Linear Regression

In [ ]:
lr_model = LinearRegression()
result = evaluate_model('Linear Regression', lr_model, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(result)
print(f"Linear Regression - R²: {result['R² Score']}, RMSE: {result['RMSE']}, MAE: {result['MAE']}")

### Ridge Regression

In [ ]:
ridge_model = Ridge(alpha=1.0)
result = evaluate_model('Ridge', ridge_model, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(result)
print(f"Ridge - R²: {result['R² Score']}, RMSE: {result['RMSE']}, MAE: {result['MAE']}")

### ElasticNet Regression

In [ ]:
en_model = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42)
result = evaluate_model('ElasticNet', en_model, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(result)
print(f"ElasticNet - R²: {result['R² Score']}, RMSE: {result['RMSE']}, MAE: {result['MAE']}")

## 7. Model Comparison

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('R² Score', ascending=False).reset_index(drop=True)
print('Model Performance Summary:')
print(results_df.to_string(index=False))

In [ ]:
# Visualize R² Scores
plt.figure(figsize=(12, 6))
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(results_df))]
bars = plt.bar(results_df['Model'], results_df['R² Score'], color=colors, edgecolor='black')
plt.title('R² Score Comparison Across Models', fontsize=14, fontweight='bold')
plt.xlabel('Model')
plt.ylabel('R² Score')
plt.xticks(rotation=45, ha='right')
plt.ylim(0.5, 1.0)
for bar, val in zip(bars, results_df['R² Score']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize RMSE
plt.figure(figsize=(12, 6))
colors = ['gold' if i == 0 else 'coral' for i in range(len(results_df))]
bars = plt.bar(results_df['Model'], results_df['RMSE'], color=colors, edgecolor='black')
plt.title('RMSE Comparison Across Models', fontsize=14, fontweight='bold')
plt.xlabel('Model')
plt.ylabel('RMSE ($)')
plt.xticks(rotation=45, ha='right')
for bar, val in zip(bars, results_df['RMSE']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             f'{val:,.0f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

## 8. Best Model: XGBRegressor

In [ ]:
best_model = xgb_model
best_result = results_df.iloc[0]
print(f'Best Model: {best_result["Model"]}')
print(f'R² Score: {best_result["R² Score"]}')
print(f'RMSE: ${best_result["RMSE"]:,.2f}')
print(f'MAE: ${best_result["MAE"]:,.2f}')

In [ ]:
# Actual vs Predicted plot for best model
y_pred_best = best_model.predict(X_test)

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_best, alpha=0.5, color='steelblue', edgecolors='white', linewidth=0.5)
min_val = min(y_test.min(), y_pred_best.min())
max_val = max(y_test.max(), y_pred_best.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
plt.title('XGBRegressor: Actual vs Predicted Salary', fontsize=14, fontweight='bold')
plt.xlabel('Actual Salary ($)')
plt.ylabel('Predicted Salary ($)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance for XGBRegressor
feature_importance = pd.Series(best_model.feature_importances_, index=X.columns)
feature_importance = feature_importance.sort_values(ascending=True)

plt.figure(figsize=(8, 5))
feature_importance.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Feature Importance - XGBRegressor', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 9. Conclusion

### Model Performance Summary

| Model | R² Score | RMSE | MAE |
|-------|----------|------|-----|
| **XGBRegressor** | **0.954072** | **11297.38** | **6156.61** |
| Extra Tree | 0.948234 | 11993.86 | 5252.84 |
| Decision Tree | 0.947939 | 12027.96 | 5269.94 |
| KNeighborsRegressor | 0.941537 | 12746.16 | 6265.76 |
| Gradient Boosting | 0.897086 | 16911.19 | 12241.72 |
| Lasso | 0.761792 | 25728.58 | 19804.42 |
| Linear Regression | 0.761772 | 25729.64 | 19811.69 |
| Ridge | 0.761743 | 25731.22 | 19803.13 |
| ElasticNet | 0.714184 | 28182.59 | 22498.07 |

### Key Findings

1. **XGBRegressor** achieves the best performance with an R² score of **0.954**, meaning it explains ~95.4% of the variance in salary.
2. **Tree-based ensemble methods** (XGBoost, Extra Trees, Decision Tree, KNN) significantly outperform linear models.
3. **Linear models** (Linear Regression, Ridge, Lasso, ElasticNet) struggle because the relationship between features and salary is non-linear.
4. **Years of Experience** and **Job Title** are the most important features for salary prediction.
